# 1- 用卷积做类激活图

在第一部分，我们将按照论文 [Learning Deep Features for Discriminative Localization](http://cnnlocalization.csail.mit.edu/) 中的描述编写类激活图（Class Activation Map，CAM）。

论文有一个配套的 GitHub 仓库：
https://github.com/zhoubolei/CAM

甚至还有一个 PyTorch 演示：
https://github.com/zhoubolei/CAM/blob/master/pytorch_CAM.py

下面的代码改编自这个演示，但我们不用 hooks，只用卷积……


In [ ]:
import io
import requests
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.nn import functional as F
import torch.optim as optim
import numpy as np
import cv2
import pdb
from matplotlib.pyplot import imshow


# 输入图像
LABELS_URL = 'https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json'
IMG_URL = 'http://media.mlive.com/news_impact/photo/9933031-large.jpg'

和演示里一样，我们使用 Resnet18 架构。为了得到 CAM，我们需要把这个网络改造成全卷积网络：在所有层，我们都要处理图像，也就是形状为 $\text{通道数} \times W\times H$ 的数据。特别地，我们关心最后一层图像，如下所示：
![](https://camo.githubusercontent.com/fb9a2d0813e5d530f49fa074c378cf83959346f7/687474703a2f2f636e6e6c6f63616c697a6174696f6e2e637361696c2e6d69742e6564752f6672616d65776f726b2e6a7067)

由于我们处理的是 Resnet18 架构，在应用 `AdaptiveAvgPool2d` 之前得到的图像大小为 $512\times 7 \times 7$（前提是输入大小为 $3\times 224\times 224 $）：
![resnet_Archi](https://pytorch.org/assets/images/resnet.png)

A- 第一件要做的事，是'去掉' resnet18 模型最后那些名为 `(avgpool)` 和 `(fc)` 的层。验证一下：对于一张大小为 $3\times 224\times 224 $ 的原始图像，你应该得到一张 $512\times 7\times 7$ 的图像。

B- 然后你需要取出 `fc` 层的权重（和偏置），也就是一个 $1000\times 512$ 的矩阵，它把 512 维向量变成 1000 维向量来做预测。然后你需要用这些权重和偏置做逐像素的应用，把你的 $512\times 7\times 7$ 图像变成 $1000\times 7\times 7$ 的输出（提示：用卷积）。这个输出可以这样解释：`output[i,j,k]` 是'像素' `[j,k]` 属于类别 `i` 的 logit。

C- 从这个 $1000\times 7\times 7$ 的输出出发，验证一下：用 `AdaptiveAvgPool2d` 就能恢复出 `resnet18` 给出的原始输出。你能理解为什么是这样吗？

D- 另外，你可以构造类激活图。画出类别 mountain bike（山地车）和 lakeside（湖边）的激活图。


## 验证：
1. 确保运行你的 notebook 时，同时显示 mountain bike 和 lakeside 两个类别的 CAM。
2. 对于上面的问题 B，你用了什么卷积？把你的答案写在这里，也就是 Pytorch 层的名称和正确的参数（in_channel、kernel……）：

<span style="color:red">用你的答案替换</span>
3. 简要解释为什么你的网络和原始 `resnet18` 给出同样的预测：

<span style="color:red">用你的答案替换</span>
4. 你的网络在不是 $224\times 224$ 尺寸（也就是不 resize）的图片上能工作吗？`resnet18` 呢？解释原因？

<span style="color:red">用你的答案替换</span>


In [ ]:
net = models.resnet18(pretrained=True)

In [ ]:
net.eval()

In [ ]:
x = torch.randn(5, 3, 224, 224)
y = net(x)
y.shape

In [ ]:
n_mean = [0.485, 0.456, 0.406]
n_std = [0.229, 0.224, 0.225]

normalize = transforms.Normalize(
   mean=n_mean,
   std=n_std
)
preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   normalize
])

# 显示我们将要使用的图像。
response = requests.get(IMG_URL)
img_pil = Image.open(io.BytesIO(response.content))
imshow(img_pil);

In [ ]:
img_tensor = preprocess(img_pil)
net = net.eval()
logit = net(img_tensor.unsqueeze(0))

In [ ]:
logit.shape

In [ ]:
img_tensor.shape

In [ ]:
# 下载 imagenet 类别列表
classes = {int(key):value for (key, value)
          in requests.get(LABELS_URL).json().items()}


def print_preds(logit):
    # 从 logit 打印预测及其'概率'
    h_x = F.softmax(logit, dim=1).data.squeeze()
    probs, idx = h_x.sort(0, True)
    probs = probs.numpy()
    idx = idx.numpy()
    # 输出预测
    for i in range(0, 5):
        print('{:.3f} -> {}'.format(probs[i], classes[idx[i]]))
    return idx

In [ ]:
idx = print_preds(logit)

In [ ]:
def returnCAM(feature_conv, idx):
    # 输入：维度为 1000*W*H 的张量 feature_conv，以及 0 到 999 之间的 idx
    # 输出：W*H 的图像，条目缩放到 0 到 255 之间用于显示
    cam = feature_conv[idx].detach().numpy()
    cam = cam - np.min(cam)
    cam_img = cam / np.max(cam)
    cam_img = np.uint8(255 * cam_img)
    return cam_img

In [ ]:
#一些工具函数
def pil_2_np(img_pil):
    # 把 PIL 图像转换成 numpy 数组
    return np.asarray(img_pil)

def display_np(img_np):
    imshow(Image.fromarray(np.uint8(img_np)))
    
def plot_CAM(img_np, CAM):
    height, width, _ = img_np.shape
    heatmap = cv2.applyColorMap(cv2.resize(CAM,(width, height)), cv2.COLORMAP_JET)
    result = heatmap * 0.3 + img_np * 0.5
    display_np(result)

In [ ]:
# 这里用一个假例子来看看事情是怎么运作的
img_np = pil_2_np(img_pil)
diag_CAM = returnCAM(torch.eye(7).unsqueeze(0),0)
plot_CAM(img_np,diag_CAM)

In [ ]:
# 在这里写你新网络的代码
net_conv = 
# 别忘了：
net_conv = net_conv.eval()

In [ ]:
# 验证一下是否正确
x = torch.randn(5, 3, 224, 224)
y = net_conv(x)
y.shape

In [ ]:
logit_conv = net_conv(img_tensor.unsqueeze(0))

In [ ]:
logit_conv.shape

In [ ]:
# 用 AdaptiveAvgPool2d 把它转成 [1,1000] 张量
logit_new = 

In [ ]:
idx = print_preds(logit_new)

In [ ]:
i = #i = #lakeside 的索引
CAM1 = returnCAM(logit_conv.squeeze(),idx[i])
plot_CAM(img_np,CAM1)

In [ ]:
i = #i = #mountain bike 的索引
CAM2 = returnCAM(logit_conv.squeeze(),idx[i])
plot_CAM(img_np,CAM2)

# 2- 对抗样本


在第二部分，我们将研究[对抗样本](https://arxiv.org/abs/1607.02533)："对抗样本是一种经过非常轻微修改的输入数据样本，这种修改旨在让机器学习分类器对它分类错误。在很多情况下，这些修改可以细微到人类观察者完全注意不到，但分类器仍然会犯错。对抗样本带来安全隐患，因为它们可以被用来对机器学习系统发起攻击……"

游戏规则：
- 攻击者不能修改分类器，也就是不能修改神经网络以及图片送入网络前的预处理。
- 即使攻击者不能修改分类器，我们假设攻击者知道分类器的架构。这里我们仍然用 `resnet18` 和标准的 Imagenet 归一化。
- 攻击者只能修改送入网络的物理图像。
- 攻击者应该骗过分类器，也就是对被破坏图像得到的标签，不应该和原始图像预测出的标签相同。

首先，你将实现 *快速梯度符号法（FGSM）*，它在[物理世界中的对抗样本](https://arxiv.org/abs/1607.02533)的 2.1 节有描述。思路很简单：假设你有一张图像 $\mathbf{x}$，把它送进网络后得到'真实'标签 $y$。你知道网络是通过关于网络参数 $\theta$ 最小化损失 $J(\mathbf{\theta}, \mathbf{x}, y)$ 来训练的。现在 $\theta$ 固定了，因为你不能修改分类器，所以你需要修改 $\mathbf{x}$。为此，你可以计算损失关于 $\mathbf{x}$ 的梯度，也就是 $\nabla_{\mathbf{x}} J(\mathbf{\theta}, \mathbf{x}, y)$，并按下面的方式使用它得到修改后的图像 $\tilde{\mathbf{x}}$：
$$
\tilde{\mathbf{x}} = \text{Clamp}\left(\mathbf{x} + \epsilon *
\text{sign}(\nabla_{\mathbf{x}} J(\mathbf{\theta}, \mathbf{x}, y)),0,1\right),
$$
其中 $\text{Clamp}(\cdot, 0,1)$ 确保 $\tilde{\mathbf{x}}$ 是一张合法的图像。
注意，如果不用 sign 而是用完整的梯度，那你就是在沿着梯度方向走，也就是增大损失 $J(\mathbf{\theta}, \mathbf{x}, y)$，从而让 $y$ 更不可能成为预测标签。


## 验证：
1. 实现这个攻击。确保显示被破坏的图像。

2. epsilon 取什么值时你的攻击能成功？这时预测的类别是什么？

<span style="color:red">用你的答案替换</span>

3. 画出梯度的符号，并让这张图通过网络。你得到什么预测？和 [Explaining and Harnessing Adversarial Examples](https://arxiv.org/abs/1412.6572) 对比一下

<span style="color:red">用你的答案替换</span>


In [ ]:
# 受到攻击的图像！
url_car = 'https://cdn130.picsart.com/263132982003202.jpg?type=webp&to=min&r=640'
response = requests.get(url_car)
img_pil = Image.open(io.BytesIO(response.content))
imshow(img_pil);

In [ ]:
# 和上面一样
preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   normalize
])

for p in net.parameters():
    p.requires_grad = False
    
x = preprocess(img_pil).clone().unsqueeze(0)
logit = net(x)

In [ ]:
_ = print_preds(logit)

In [ ]:
t_std = torch.from_numpy(np.array(n_std, dtype=np.float32)).view(-1, 1, 1)
t_mean = torch.from_numpy(np.array(n_mean, dtype=np.float32)).view(-1, 1, 1)

def plot_img_tensor(img):
    imshow(np.transpose(img.detach().numpy(), [1,2,0]))

def plot_untransform(x_t): 
    x_np = (x_t * t_std + t_mean).detach().numpy()
    x_np = np.transpose(x_np, [1, 2, 0])
    imshow(x_np)

In [ ]:
# 这里显示一张以张量形式给出的图像
x_img = (x * t_std + t_mean).squeeze(0)
plot_img_tensor(x_img)

In [ ]:
# 你的攻击实现
def fgsm_attack(image, epsilon, data_grad):
    # 收集数据梯度的逐元素符号
    
    # 通过调整输入图像的每个像素来创建扰动图像
    
    # 加上裁剪，保持 [0,1] 范围
    
    # 返回扰动后的图像
    return perturbed_image

In [ ]:
idx = 656 #idx = 656 #minivan
criterion = nn.CrossEntropyLoss()
x_img.requires_grad = True
logit = net(normalize(x_img).unsqueeze(0))
target = torch.tensor([idx])

    #TODO：计算要反向传播的损失

_ = print_preds(logit)

In [ ]:
# 在这里进行你的攻击
epsilon = 0
x_att = fgsm_attack(x_img,epsilon,?)

In [ ]:
# 被破坏图像的新预测
logit = net(normalize(x_att).unsqueeze(0))
_ = print_preds(logit)

In [ ]:
# 你能看出区别吗？
plot_img_tensor(x_att)

In [ ]:
# 别忘了画出梯度的符号
gradient = 
plot_img_tensor((1+gradient)/2)

In [ ]:
# 对这个梯度，预测是什么？
logit = net(normalize(gradient).unsqueeze(0))
_ = print_preds(logit)

# 3- 把汽车变成猫

现在我们实现 *迭代目标类别法（ITCM）*，定义见[对抗攻击与防御竞赛](https://arxiv.org/abs/1804.00097)的公式 (4)。

为了测试它，我们把汽车（被我们的 `resnet18` 标记为 minivan，小货车）变成一只[虎斑猫](https://en.wikipedia.org/wiki/Tabby_cat)（Imagenet 中的类别 281）。当然你也可以试其他目标。


## 验证：
1. 实现 ITCM，并确保显示得到的图像。


In [ ]:
x = preprocess(img_pil).clone()
xd = preprocess(img_pil).clone()
xd.requires_grad = True

In [ ]:
idx = 281 #idx = 281 #tabby
optimizer = optim.SGD([xd], lr=0.01)

for i in range(200):
    #TODO：在这里写你的代码
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(loss.item())
    
    _ = print_preds(output)
    print(i,'-----------------')
    
    # TODO：一旦满意就跳出循环
    if ?:
        break

In [ ]:
_ = print_preds(output)

In [ ]:
# 画出被破坏的图像


# 4- 猫藏在哪里？

最后，我们用 CAM 来理解网络在图像里哪里看到了猫。


## 验证：
1. 显示类别 tabby 的 CAM

2. 显示类别 minivan 的 CAM

3. 猫在哪里？
